# Klasifikasi Sampah dengan MobileNetV4-Small → `.tflite`

Notebook ini melakukan *fine-tuning* **MobileNetV4-Conv-Small** (bobot pretrained ImageNet dari `timm`) pada dataset
[Garbage Classification](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification)
(6 kelas: `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`), lalu mengekspornya ke format **`.tflite`** (LiteRT)
untuk dipakai di Android / iOS / Flutter.

**Cara pakai**
1. `Runtime` → `Change runtime type` → pilih **GPU** (mis. T4).
2. Jalankan sel dari atas ke bawah. Saat sel upload muncul, pilih file **`archive.zip`** dari komputer Anda.
3. Di akhir, `garbage_mobilenetv4_small.tflite` dan `labels.txt` otomatis terunduh.

**Alur:** instalasi → upload & ekstrak zip → siapkan data (train/val/test) → latih → evaluasi → ekspor ke TFLite → verifikasi file `.tflite` → unduh.

## 1. Instalasi

- `timm` menyediakan MobileNetV4 beserta bobot pretrained-nya.
- `litert-torch` (penerus `ai-edge-torch`) mengonversi model PyTorch menjadi `.tflite`.

> Jika setelah instalasi Colab meminta **Restart session** (atau sel berikutnya error saat impor), pilih **Runtime → Restart session**, lalu lanjutkan dari sel berikutnya — sel instalasi tidak perlu dijalankan lagi. Pesan merah soal *dependency conflict* dari pip biasanya aman diabaikan.

In [ ]:
# torchaudio tidak dipakai di notebook ini. Dihapus agar tidak bentrok versi CUDA
# kalau pip mengganti versi torch saat memasang litert-torch.
!pip uninstall -y -q torchaudio
!pip install -q timm litert-torch torchvision ai-edge-litert

## 2. Impor library & cek GPU

In [ ]:
import os, sys, time, json, copy, random, hashlib, zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # toleran terhadap file JPEG yang sedikit rusak

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms as T
    import timm
    import litert_torch
except Exception as e:
    raise RuntimeError(
        "Gagal mengimpor library. Pilih Runtime > Restart session, lalu jalankan ulang "
        "mulai dari sel ini (sel instalasi tidak perlu diulang).\nDetail: %r" % (e,)
    )

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
if DEVICE.type == "cuda":
    print("Device :", torch.cuda.get_device_name(0))
else:
    print("Device : CPU  <- TIDAK ADA GPU! Pilih Runtime > Change runtime type > T4 GPU agar training cepat.")

## 3. Upload & ekstrak `archive.zip`

Jalankan sel di bawah, lalu klik **Choose Files** dan pilih `archive.zip` (file dataset dari Kaggle).
Jika file zip sudah ada di panel *Files* (ikon folder di kiri) pada folder `/content`, tahap upload dilewati.

In [ ]:
import shutil
from google.colab import files

# Cek dulu: apakah zip sudah ada di /content (mis. di-drag ke panel Files di kiri)?
ZIP_PATH = next(iter(sorted(Path("/content").glob("*.zip"))), None)

if ZIP_PATH is None:
    uploaded = files.upload()          # <- pilih archive.zip dari komputer Anda
    if not uploaded:
        raise RuntimeError("Tidak ada file yang diunggah. Jalankan sel ini lagi lalu pilih archive.zip.")
    ZIP_PATH = Path("/content") / next(iter(uploaded))
    del uploaded

print("File zip:", ZIP_PATH, f"({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")

# Ekstrak (hapus hasil ekstrak lama bila ada)
DATA_DIR = Path("/content/dataset")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_DIR)
print("Diekstrak ke:", DATA_DIR)

## 4. Baca dataset & buang duplikat

Sel ini mengindeks semua gambar per kelas (nama folder = label). Duplikat dideteksi lewat hash isi file (MD5).

Ini penting karena zip dataset ini berisi **dua salinan identik** dari folder yang sama (`Garbage classification` dan `garbage classification`).
Kalau tidak dibuang, gambar yang sama bisa masuk ke train *dan* test sehingga akurasi terlihat lebih bagus dari kenyataannya (kebocoran data).

In [ ]:
CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


records, seen, n_found = [], set(), 0          # records = [(path, indeks_kelas), ...]
for p in sorted(DATA_DIR.rglob("*")):
    if not p.is_file() or p.suffix.lower() not in IMG_EXT:
        continue
    if "__MACOSX" in p.parts or p.name.startswith("._"):
        continue
    cls = p.parent.name.lower()
    if cls not in CLASS_NAMES:
        continue
    n_found += 1
    digest = file_md5(p)
    if digest in seen:                          # isi file sama persis -> lewati
        continue
    seen.add(digest)
    records.append((str(p), CLASS_NAMES.index(cls)))

n_dup = n_found - len(records)
print(f"Gambar ditemukan : {n_found}")
print(f"Duplikat dibuang : {n_dup}")
print(f"Gambar unik      : {len(records)}\n")
if n_dup:
    print("Catatan: duplikat (mis. salinan ganda folder di dalam zip) dibuang supaya gambar yang sama\n"
          "tidak muncul di train sekaligus val/test.\n")

counts = np.bincount([r[1] for r in records], minlength=NUM_CLASSES)
for name, c in zip(CLASS_NAMES, counts):
    print(f"  {name:<10s}{c:>5d}")
assert (counts > 0).all(), "Ada kelas yang tidak ditemukan. Pastikan folder per-kelas ada di dalam zip."

Contoh gambar dari tiap kelas (pastikan label & gambar sudah sesuai):

In [ ]:
rng = random.Random(SEED)
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(10, 2.1 * NUM_CLASSES))
for ci, name in enumerate(CLASS_NAMES):
    items = [r for r in records if r[1] == ci]
    for j, (path, _) in enumerate(rng.sample(items, 4)):
        ax = axes[ci, j]
        ax.imshow(Image.open(path).convert("RGB"))
        ax.axis("off")
        if j == 0:
            ax.set_title(name, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Bagi data: train / val / test

Pembagian *stratified* 70% / 15% / 15% (proporsi tiap kelas dijaga, `seed` tetap).
Set **test** tidak dipakai saat training — hanya untuk evaluasi akhir.

In [ ]:
paths  = [r[0] for r in records]
labels = [r[1] for r in records]

tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
va_p, te_p, va_y, te_y   = train_test_split(tmp_p, tmp_y, test_size=0.50, stratify=tmp_y, random_state=SEED)

print(f"{'kelas':<10s}{'train':>7s}{'val':>6s}{'test':>6s}")
for ci, name in enumerate(CLASS_NAMES):
    print(f"{name:<10s}{tr_y.count(ci):>7d}{va_y.count(ci):>6d}{te_y.count(ci):>6d}")
print(f"{'TOTAL':<10s}{len(tr_y):>7d}{len(va_y):>6d}{len(te_y):>6d}")

## 6. Bangun model MobileNetV4-Small

`mobilenetv4_conv_small.e2400_r224_in1k` dari `timm` (pretrained ImageNet-1k, input 224×224).
Kepala klasifikasi otomatis diganti menjadi 6 kelas; `drop_rate=0.2` membantu mengurangi overfitting pada dataset kecil.

In [ ]:
MODEL_NAME = "mobilenetv4_conv_small.e2400_r224_in1k"
IMG_SIZE = 224

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=0.2).to(DEVICE)

# mean/std normalisasi yang dipakai saat pretraining model ini
data_cfg = timm.data.resolve_model_data_config(model)
MEAN, STD = tuple(data_cfg["mean"]), tuple(data_cfg["std"])
print("mean:", MEAN, "| std:", STD)
print(f"Jumlah parameter: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} juta")

## 7. Augmentasi & DataLoader

- **Train:** zoom/crop acak, flip horizontal & vertikal (sampah tidak punya "sisi atas" yang baku), rotasi kecil, variasi warna ringan.
- **Val/Test:** gambar langsung di-*resize* ke 224×224 (tanpa crop) — sama persis dengan yang sebaiknya dilakukan di aplikasi nanti.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_tfm = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


class GarbageDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths, self.labels, self.tfm = paths, labels, tfm

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(img), self.labels[i]


train_ds = GarbageDataset(tr_p, tr_y, train_tfm)
val_ds   = GarbageDataset(va_p, va_y, eval_tfm)
test_ds  = GarbageDataset(te_p, te_y, eval_tfm)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, persistent_workers=True, **loader_kw)
val_loader   = DataLoader(val_ds, shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds, shuffle=False, **loader_kw)
print("Batch per epoch (train):", len(train_loader))

## 8. Training

- Optimizer **AdamW**: *learning rate* lebih kecil untuk backbone (sudah pretrained) dan lebih besar untuk kepala klasifikasi yang baru, dijadwalkan dengan **OneCycle**.
- **Class weight** + *label smoothing* membantu kelas yang datanya sedikit (`trash`).
- Mixed precision (fp16) aktif otomatis di GPU. Bobot dengan akurasi validasi tertinggi disimpan dan dipakai di tahap berikutnya.

Dataset ini kecil, jadi training biasanya selesai dalam beberapa menit di GPU. Ubah `EPOCHS` bila ingin lebih lama/singkat.

In [ ]:
EPOCHS      = 20
LR_BACKBONE = 3e-4
LR_HEAD     = 1e-3

# Bobot kelas: kelas yang jarang (mis. trash) diberi bobot lebih besar
cls_count = np.bincount(tr_y, minlength=NUM_CLASSES)
class_w = torch.tensor(cls_count.sum() / (NUM_CLASSES * cls_count), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.1)

head_params = list(model.get_classifier().parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": LR_BACKBONE},
     {"params": head_params, "lr": LR_HEAD}],
    weight_decay=1e-2,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR_BACKBONE, LR_HEAD],
    epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.15,
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_epoch(loader, train):
    model.train(train)
    loss_sum, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return loss_sum / n, correct / n

In [ ]:
history = {"tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}
best = {"acc": -1.0, "loss": float("inf"), "epoch": 0, "state": None}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)

    history["tr_loss"].append(tr_loss); history["tr_acc"].append(tr_acc)
    history["va_loss"].append(va_loss); history["va_acc"].append(va_acc)

    improved = va_acc > best["acc"] or (va_acc == best["acc"] and va_loss < best["loss"])
    if improved:
        best.update(acc=va_acc, loss=va_loss, epoch=ep, state=copy.deepcopy(model.state_dict()))

    print(f"Epoch {ep:02d}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time() - t0:.0f}s"
          + ("  <- terbaik" if improved else ""))

model.load_state_dict(best["state"])
torch.save(best["state"], "best_mobilenetv4_small.pth")
print(f"\nBobot terbaik: epoch {best['epoch']} (akurasi val {best['acc']:.3f}) -> best_mobilenetv4_small.pth")

## 9. Evaluasi pada test set (model PyTorch)

In [ ]:
ep_axis = range(1, len(history["tr_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ep_axis, history["tr_loss"], label="train")
ax[0].plot(ep_axis, history["va_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep_axis, history["tr_acc"], label="train")
ax[1].plot(ep_axis, history["va_acc"], label="val")
ax[1].set_title("Akurasi"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(DEVICE)).argmax(1).cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


y_true, y_pred = predict(test_loader)
print(f"Akurasi test (PyTorch): {(y_true == y_pred).mean():.4f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## 10. Ekspor ke `.tflite`

Model dibungkus (`MobileWrapper`) agar praktis dipakai di aplikasi mobile:

- **Input:** `float32 [1, 224, 224, 3]` (NHWC, urutan RGB), nilai piksel **0–255**. Normalisasi mean/std sudah ditanam di dalam model.
- **Output:** `float32 [1, 6]` berisi **probabilitas** (softmax), urutannya sesuai `labels.txt`.

Konversi memakai `litert_torch.convert` (library resmi Google AI Edge). Prosesnya bisa memakan waktu beberapa saat.

In [ ]:
class MobileWrapper(nn.Module):
    """Pembungkus model untuk aplikasi mobile.

    Input : float32 [N, H, W, 3]       (NHWC, RGB, nilai piksel 0..255)
    Output: float32 [N, jumlah_kelas]  (probabilitas softmax)
    Normalisasi mean/std ditanam di sini, jadi aplikasi tidak perlu menormalisasi lagi.
    """

    def __init__(self, net, mean, std):
        super().__init__()
        self.net = net
        mean = torch.tensor(mean, dtype=torch.float32).view(1, 1, 1, 3)
        std = torch.tensor(std, dtype=torch.float32).view(1, 1, 1, 3)
        self.register_buffer("scale", 1.0 / (255.0 * std))   # (x/255 - mean)/std  ==  x*scale + shift
        self.register_buffer("shift", -mean / std)

    def forward(self, x):
        x = x * self.scale + self.shift        # normalisasi, masih di layout NHWC
        x = x.permute(0, 3, 1, 2)              # NHWC -> NCHW untuk backbone PyTorch
        return torch.softmax(self.net(x), dim=1)


export_net = copy.deepcopy(model).to("cpu").eval()
if hasattr(export_net, "drop_rate"):
    export_net.drop_rate = 0.0                 # buang dropout dari graph
wrapper = MobileWrapper(export_net, MEAN, STD).eval()

sample_inputs = (torch.rand(1, IMG_SIZE, IMG_SIZE, 3) * 255.0,)     # NHWC, 0..255
with torch.no_grad():
    print("Contoh output PyTorch:", wrapper(*sample_inputs)[0].numpy().round(3))

TFLITE_PATH = "garbage_mobilenetv4_small.tflite"
with torch.no_grad():
    try:
        edge_model = litert_torch.convert(wrapper, sample_inputs)
    except Exception as e:
        print("Konversi mode default gagal:", repr(e)[:400])
        print("Mencoba ulang dengan strict_export=False ...")
        edge_model = litert_torch.convert(wrapper, sample_inputs, strict_export=False)

edge_model.export(TFLITE_PATH)
print(f"\nTersimpan: {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH) / 1e6:.2f} MB)")

## 11. Verifikasi file `.tflite`

File `.tflite` yang baru dibuat dijalankan dengan interpreter LiteRT, lalu hasilnya dibandingkan dengan model PyTorch
pada seluruh test set. Ini memastikan hasil konversi akurat sebelum dipakai di aplikasi.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

interp = Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
in_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
print("Input :", list(in_d["shape"]), in_d["dtype"].__name__)
print("Output:", list(out_d["shape"]), out_d["dtype"].__name__)


def load_array(path):
    """Preprocessing yang sama dengan di aplikasi: RGB -> resize 224x224 -> float32 0..255."""
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)[None]          # [1, 224, 224, 3]


def tflite_predict(arr):
    interp.set_tensor(in_d["index"], arr)
    interp.invoke()
    return interp.get_tensor(out_d["index"])[0].copy()


t0 = time.time()
probs_tfl, probs_pt = [], []
with torch.no_grad():
    for path in te_p:
        arr = load_array(path)
        probs_tfl.append(tflite_predict(arr))
        probs_pt.append(wrapper(torch.from_numpy(arr))[0].numpy())
probs_tfl, probs_pt = np.stack(probs_tfl), np.stack(probs_pt)
pred_tfl, pred_pt = probs_tfl.argmax(1), probs_pt.argmax(1)
y_te = np.array(te_y)

same = (pred_pt == pred_tfl).mean()
print(f"\n{len(te_p)} gambar test diproses dalam {time.time() - t0:.1f} detik")
print(f"Akurasi test - PyTorch (CPU) : {(pred_pt == y_te).mean() * 100:.2f}%")
print(f"Akurasi test - TFLite        : {(pred_tfl == y_te).mean() * 100:.2f}%")
print(f"Prediksi PyTorch = TFLite    : {same * 100:.2f}%")
print(f"Selisih probabilitas terbesar: {np.abs(probs_pt - probs_tfl).max():.2e}")
if same < 0.99:
    print("PERINGATAN: hasil TFLite berbeda cukup jauh dari PyTorch. Periksa langkah ekspor di atas.")

In [ ]:
idx = random.Random(1).sample(range(len(te_p)), 8)
fig, axes = plt.subplots(2, 4, figsize=(12, 6.4))
for ax, i in zip(axes.ravel(), idx):
    p = probs_tfl[i]
    k = int(p.argmax())
    ax.imshow(Image.open(te_p[i]).convert("RGB"))
    ax.axis("off")
    ax.set_title(f"{CLASS_NAMES[k]} ({p[k] * 100:.0f}%)\nasli: {CLASS_NAMES[te_y[i]]}",
                 color="green" if k == te_y[i] else "red", fontsize=10)
plt.suptitle("Prediksi model .tflite pada contoh gambar test", y=1.0)
plt.tight_layout()
plt.show()

## 12. Simpan label & unduh hasil

Dua file akan diunduh: **`garbage_mobilenetv4_small.tflite`** (model) dan **`labels.txt`** (nama kelas, urutannya = indeks output model).
Jika browser menanyakan izin mengunduh banyak file, pilih *Allow*. Kedua file juga tersedia di panel *Files* (ikon folder di kiri).

In [ ]:
from google.colab import files

LABELS_PATH = "labels.txt"
Path(LABELS_PATH).write_text("\n".join(CLASS_NAMES))

print("=" * 64)
print("MODEL SIAP DIPAKAI")
print("=" * 64)
print(f"Model  : {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH) / 1e6:.2f} MB)")
print(f"Label  : {LABELS_PATH} -> {CLASS_NAMES}")
print(f"Input  : float32 [1, {IMG_SIZE}, {IMG_SIZE}, 3], RGB, nilai piksel 0..255 (tanpa normalisasi tambahan)")
print(f"Output : float32 [1, {NUM_CLASSES}], probabilitas softmax sesuai urutan label")
print(f"Akurasi test (TFLite): {(pred_tfl == y_te).mean() * 100:.2f}%")

files.download(TFLITE_PATH)
files.download(LABELS_PATH)

## Selesai 🎉

**Memakai model di aplikasi**
1. Ambil gambar (kamera/galeri) → pastikan RGB → **resize langsung ke 224×224** (tanpa crop).
2. Isi tensor input `float32 [1, 224, 224, 3]` dengan nilai piksel **0–255** apa adanya (jangan dibagi 255, jangan dikurangi mean/std — sudah ada di dalam model).
3. Jalankan model, ambil indeks probabilitas tertinggi, lalu cocokkan dengan baris yang sama di `labels.txt`.

Gunakan library LiteRT/TFLite versi terbaru di aplikasi agar semua operasi pada model didukung.